# RT-DETR Benchmark — OOD Dataset (17 classes)

**Features:**
- 5 weak classes removed (person, tree, street_light, pole, traffic_light)
- SAHI sliced inference for small-object recall
- Negative sample accuracy tracking
- Full per-class AP heatmap
- Auto-resume on Kaggle crash

**Models:** rtdetrn, rtdetrs, rtdetrm

In [ ]:
# ═══ CELL 1: Setup & GPU Check ═══
import shutil, os
shutil.rmtree('/kaggle/working/datasets', ignore_errors=True)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)}  VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

In [ ]:
# ═══ CELL 2: Install & Prepare Dataset (remove 5 weak classes) ═══
!pip install -q ultralytics sahi

import yaml, shutil
from pathlib import Path

# ── Auto-find dataset ──
input_root = Path('/kaggle/input')
yaml_candidates = list(input_root.rglob('data.yaml'))
if not yaml_candidates:
    raise FileNotFoundError('No data.yaml found under /kaggle/input!')

src_yaml = yaml_candidates[0]
orig_dataset = src_yaml.parent
print(f'Found dataset at: {orig_dataset}')

# ── Classes to REMOVE ──
REMOVE_CLASSES = {'person', 'tree', 'street_light', 'pole', 'traffic_light'}

with open(src_yaml) as f:
    orig_cfg = yaml.safe_load(f)

old_names = orig_cfg['names']
if isinstance(old_names, dict):
    old_names = [old_names[i] for i in sorted(old_names.keys())]

# Build remapping: old_id -> new_id (only for kept classes)
REMOVE_IDS = {i for i, name in enumerate(old_names) if name in REMOVE_CLASSES}
kept = [(i, name) for i, name in enumerate(old_names) if name not in REMOVE_CLASSES]
OLD_TO_NEW = {old_id: new_id for new_id, (old_id, _) in enumerate(kept)}
new_names = [name for _, name in kept]

print(f'Original: {len(old_names)} classes')
print(f'Removed:  {REMOVE_CLASSES}')
print(f'Keeping:  {len(new_names)} classes: {new_names}')

# ── Copy dataset and remap labels ──
clean_dir = Path('/kaggle/working/dataset_17cls')
if clean_dir.exists():
    print(f'Clean dataset already exists at {clean_dir}, skipping copy.')
else:
    print('Copying and remapping dataset...')
    for split in ['train', 'valid', 'test']:
        # Copy images
        src_imgs = orig_dataset / split / 'images'
        dst_imgs = clean_dir / split / 'images'
        if src_imgs.exists():
            shutil.copytree(src_imgs, dst_imgs)
            print(f'  {split}/images: {len(list(dst_imgs.glob("*")))} files')

        # Remap labels
        src_lbls = orig_dataset / split / 'labels'
        dst_lbls = clean_dir / split / 'labels'
        dst_lbls.mkdir(parents=True, exist_ok=True)
        if src_lbls.exists():
            remapped, skipped = 0, 0
            for lbl_file in src_lbls.glob('*.txt'):
                lines = lbl_file.read_text().strip().splitlines()
                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    old_cls = int(parts[0])
                    if old_cls in REMOVE_IDS:
                        skipped += 1
                        continue
                    new_cls = OLD_TO_NEW[old_cls]
                    new_lines.append(f'{new_cls} {" ".join(parts[1:])}')
                (dst_lbls / lbl_file.name).write_text('\n'.join(new_lines) + ('\n' if new_lines else ''))
                remapped += 1
            print(f'  {split}/labels: {remapped} files remapped, {skipped} annotations removed')

# ── Write clean data.yaml ──
clean_yaml = '/kaggle/working/data_17cls.yaml'
clean_cfg = {
    'path': str(clean_dir),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(new_names),
    'names': new_names,
}
with open(clean_yaml, 'w') as f:
    yaml.dump(clean_cfg, f)
print(f'\n✅ Clean YAML -> {clean_yaml} ({len(new_names)} classes)')

In [ ]:
# ═══ CELL 3: Configuration ═══
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np
import time
import gc, torch, os, yaml

def _safe_cuda_empty_cache():
    if not torch.cuda.is_available(): return
    try: torch.cuda.synchronize()
    except: pass
    try: torch.cuda.empty_cache()
    except: pass

# ┌────────────────────────────────────────────────┐
# │  CONFIGURATION                                 │
# └────────────────────────────────────────────────┘
DATA_YAML    = '/kaggle/working/data_17cls.yaml'
IMG_SIZE     = 640
EPOCHS       = 100
BATCH        = 16
DEVICE       = 0
PATIENCE     = 50
WORKERS      = 4
AMP          = True

RESULTS_CSV  = 'benchmark_rtdetr.csv'
PERCLASS_CSV = 'benchmark_rtdetr_perclass.csv'

# SAHI settings
USE_SAHI          = True
SAHI_SLICE_SIZE   = 320
SAHI_OVERLAP      = 0.2

CLASS_NAMES = [
    'bench', 'bicycle', 'bus', 'bus_stop', 'car', 'crutch', 'curb', 'dog',
    'fire_hydrant', 'motorcycle', 'spherical_roadblock', 'stairs', 'stop_sign',
    'train', 'truck', 'warning_column', 'waste_container'
]

MODELS = [
    'rtdetr-l.pt',
    'rtdetr-x.pt',
]

print(f'Config: device={DEVICE}, batch={BATCH}, epochs={EPOCHS}, imgsz={IMG_SIZE}')
print(f'Classes: {len(CLASS_NAMES)}')
print(f'SAHI: {USE_SAHI} (slice={SAHI_SLICE_SIZE})')
print(f'Models: {MODELS}')

In [ ]:
# ═══ CELL 4: Benchmark Function ═══
import cv2
try:
    from sahi.predict import get_sliced_prediction
    from sahi import AutoDetectionModel
    SAHI_AVAILABLE = True
except ImportError:
    SAHI_AVAILABLE = False
    print('⚠️ SAHI not available, falling back to standard inference')

def benchmark_model(model_name):
    print(f"\n{'='*60}")
    print(f'  BENCHMARK: {model_name}')
    print(f"{'='*60}")

    run_name   = f"{model_name.replace('.pt', '')}_ood_17cls"
    run_dir    = Path(f'/kaggle/working/runs/detect/{run_name}')
    best_pt    = run_dir / 'weights' / 'best.pt'
    last_pt    = run_dir / 'weights' / 'last.pt'
    results_p  = run_dir / 'results.csv'

    already_done = results_p.exists() and best_pt.exists()
    use_resume   = last_pt.exists() and not already_done

    # ── 1. Train ──
    if already_done:
        print('✅ Training already complete — skipping to validation')
    elif use_resume:
        print(f'🔁 Resuming from {last_pt}')
        model = YOLO(str(last_pt))
        model.train(resume=True)
        del model; gc.collect(); _safe_cuda_empty_cache()
    else:
        print('🚀 Starting fresh training')
        model = YOLO(model_name)
        model.train(
            data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
            device=DEVICE, workers=WORKERS, amp=AMP,
            optimizer='auto', cos_lr=True, lr0=0.01, lrf=0.01,
            patience=PATIENCE, weight_decay=0.0005,
            mosaic=1.0, mixup=0.1, copy_paste=0.1, degrees=10.0,
            translate=0.1, scale=0.5, fliplr=0.5,
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, erasing=0.4,
            name=run_name, exist_ok=True, save=True, plots=True,
        )
        del model; gc.collect(); _safe_cuda_empty_cache()

    # ── 2. Validate on test set ──
    if not best_pt.exists():
        raise FileNotFoundError(f'best.pt not found at {best_pt}')

    best_model = YOLO(str(best_pt))
    metrics = best_model.val(
        data=DATA_YAML, split='test',
        device=DEVICE, imgsz=IMG_SIZE, workers=WORKERS,
    )

    # ── 3. Speed + SAHI + Negative Samples ──
    with open(DATA_YAML) as f:
        dcfg = yaml.safe_load(f)
    test_img_dir = Path(dcfg['path']) / dcfg['test']
    test_lbl_dir = Path(str(test_img_dir).replace('images', 'labels'))
    _ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    test_images = [p for p in test_img_dir.glob('*.*') if p.suffix.lower() in _ext][:200]
    if not test_images:
        raise FileNotFoundError(f'No test images under {test_img_dir}')

    # SAHI model
    sahi_model = None
    if USE_SAHI and SAHI_AVAILABLE:
        sahi_model = AutoDetectionModel.from_pretrained(
            model_type='ultralytics', model_path=str(best_pt),
            device=f'cuda:{DEVICE}' if torch.cuda.is_available() else 'cpu',
            confidence_threshold=0.25,
        )

    latencies = []
    tn_count, fp_count, neg_total = 0, 0, 0

    # Warmup
    for _ in range(3):
        best_model(str(test_images[0]), imgsz=IMG_SIZE, device=DEVICE, verbose=False)

    for img_path in test_images:
        t0 = time.perf_counter()
        if sahi_model:
            res = get_sliced_prediction(
                str(img_path), sahi_model,
                slice_height=SAHI_SLICE_SIZE, slice_width=SAHI_SLICE_SIZE,
                overlap_height_ratio=SAHI_OVERLAP, overlap_width_ratio=SAHI_OVERLAP,
                verbose=0,
            )
            boxes_count = len(res.object_prediction_list)
        else:
            res = best_model(str(img_path), imgsz=IMG_SIZE, device=DEVICE, verbose=False)
            boxes_count = len(res[0].boxes) if res and res[0].boxes is not None else 0
        latencies.append((time.perf_counter() - t0) * 1000)

        # Negative sample check
        lbl_p = test_lbl_dir / (img_path.stem + '.txt')
        is_neg = not lbl_p.exists() or lbl_p.stat().st_size == 0
        if is_neg:
            neg_total += 1
            if boxes_count == 0: tn_count += 1
            else: fp_count += 1

    neg_acc = tn_count / neg_total if neg_total > 0 else 1.0

    # ── 4. Collect metrics ──
    size_mb  = best_pt.stat().st_size / 1e6
    params_m = sum(p.numel() for p in best_model.model.parameters()) / 1e6
    p = float(metrics.box.mp)
    r = float(metrics.box.mr)
    f1 = 2*p*r/(p+r) if (p+r) > 0 else 0.0

    row = {
        'model':        model_name.replace('.pt', ''),
        'mAP@0.5':      round(float(metrics.box.map50), 4),
        'mAP@0.5:0.95': round(float(metrics.box.map),   4),
        'precision':    round(p,  4),
        'recall':       round(r,  4),
        'F1':           round(f1, 4),
        'speed_ms/img': round(float(np.mean(latencies)), 2),
        'neg_acc':      round(neg_acc, 4),
        'neg_total':    neg_total,
        'sahi':         bool(sahi_model),
        'size_MB':      round(size_mb,  1),
        'params_M':     round(params_m, 1),
    }

    per_class = {}
    for i, name in enumerate(CLASS_NAMES):
        if i < len(metrics.box.ap50):
            per_class[name] = round(float(metrics.box.ap50[i]), 4)

    del best_model; gc.collect(); _safe_cuda_empty_cache()
    return row, per_class

In [ ]:
# ═══ CELL 5: Run Benchmark ═══
rows = []
all_per_class = {}

for model_name in MODELS:
    try:
        row, per_class = benchmark_model(model_name)
        rows.append(row)
        all_per_class[row['model']] = per_class
        print(f"\n  ┌─────────────────────────────────┐")
        print(f"  │ {row['model']:^31s} │")
        print(f"  ├─────────────────────────────────┤")
        print(f"  │ mAP@0.5      : {row['mAP@0.5']:<15} │")
        print(f"  │ mAP@0.5:0.95 : {row['mAP@0.5:0.95']:<15} │")
        print(f"  │ Precision    : {row['precision']:<15} │")
        print(f"  │ Recall       : {row['recall']:<15} │")
        print(f"  │ F1           : {row['F1']:<15} │")
        print(f"  │ Neg Accuracy : {row['neg_acc']:<15} │")
        print(f"  │ Speed        : {row['speed_ms/img']} ms/img       │")
        print(f"  │ SAHI         : {row['sahi']!s:<15} │")
        print(f"  │ Size         : {row['size_MB']} MB            │")
        print(f"  │ Params       : {row['params_M']} M             │")
        print(f"  └─────────────────────────────────┘")
    except Exception as e:
        print(f'  ❌ SKIPPED {model_name}: {e}')
        import traceback; traceback.print_exc()
        gc.collect(); _safe_cuda_empty_cache()

# Save CSVs
_cols = [
    'model', 'mAP@0.5', 'mAP@0.5:0.95', 'precision', 'recall', 'F1',
    'speed_ms/img', 'neg_acc', 'neg_total', 'sahi', 'size_MB', 'params_M',
]
df = pd.DataFrame(rows, columns=_cols) if rows else pd.DataFrame(columns=_cols)
df.to_csv(RESULTS_CSV, index=False)

if all_per_class:
    df_pc = pd.DataFrame(all_per_class).T
else:
    df_pc = pd.DataFrame(columns=CLASS_NAMES)
df_pc.index.name = 'model'
df_pc.to_csv(PERCLASS_CSV)

print(f'\n✅ Saved -> {RESULTS_CSV} ({len(df)} rows)')
print(f'✅ Saved -> {PERCLASS_CSV}')

In [ ]:
# ═══ CELL 6: Results Table ═══
from IPython.display import display

df = pd.read_csv(RESULTS_CSV)
if not df.empty:
    print('=' * 60)
    print('  RT-DETR BENCHMARK — 17 classes (SAHI + Neg Samples)')
    print('=' * 60)
    styled = (
        df.style
        .set_caption('RT-DETR Benchmark — 17 Classes')
        .format({
            'mAP@0.5': '{:.4f}', 'mAP@0.5:0.95': '{:.4f}',
            'precision': '{:.4f}', 'recall': '{:.4f}', 'F1': '{:.4f}',
            'neg_acc': '{:.4f}', 'speed_ms/img': '{:.1f} ms',
            'size_MB': '{:.1f} MB', 'params_M': '{:.1f} M',
        })
        .highlight_max(subset=['mAP@0.5', 'mAP@0.5:0.95', 'precision', 'recall', 'F1', 'neg_acc'], color='#2d6a2e')
        .highlight_min(subset=['speed_ms/img', 'size_MB', 'params_M'], color='#1a5276')
        .set_properties(**{'text-align': 'center', 'font-size': '13px'})
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold'), ('padding', '10px')]},
            {'selector': 'th', 'props': [('background-color', '#1a1a2e'), ('color', 'white'), ('padding', '8px')]},
        ])
        .hide(axis='index')
    )
    display(styled)

In [ ]:
# ═══ CELL 7: Per-Class Heatmap ═══
df_pc = pd.read_csv(PERCLASS_CSV, index_col=0)
if not df_pc.empty:
    print('Per-class mAP@0.5 across RT-DETR variants')
    print('-' * 60)
    styled_pc = (
        df_pc.style
        .set_caption('Per-Class mAP@0.5 — RT-DETR (17 classes)')
        .format('{:.4f}')
        .background_gradient(cmap='RdYlGn', axis=None, vmin=0, vmax=1)
        .set_properties(**{'text-align': 'center', 'font-size': '12px'})
        .set_table_styles([
            {'selector': 'caption', 'props': [('font-size', '15px'), ('font-weight', 'bold')]},
            {'selector': 'th', 'props': [('background-color', '#1a1a2e'), ('color', 'white'), ('font-size', '11px'), ('padding', '6px')]},
        ])
    )
    display(styled_pc)

In [ ]:
# ═══ CELL 8: Charts ═══
import matplotlib.pyplot as plt

df = pd.read_csv(RESULTS_CSV)
if not df.empty:
    fig, axes = plt.subplots(1, 4, figsize=(22, 5))
    axes[0].barh(df['model'], df['mAP@0.5'], color='#2d6a2e')
    axes[0].set_xlabel('mAP@0.5'); axes[0].set_title('Accuracy'); axes[0].set_xlim(0, 1)
    axes[1].barh(df['model'], df['speed_ms/img'], color='#1a5276')
    axes[1].set_xlabel('ms/img'); axes[1].set_title('Speed')
    axes[2].barh(df['model'], df['neg_acc'], color='#8e44ad')
    axes[2].set_xlabel('Accuracy'); axes[2].set_title('Negative Sample Acc'); axes[2].set_xlim(0, 1)
    axes[3].barh(df['model'], df['size_MB'], color='#c0392b')
    axes[3].set_xlabel('MB'); axes[3].set_title('Model Size')
    plt.suptitle('RT-DETR Benchmark — 17 Classes (SAHI)', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('benchmark_rtdetr_chart.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ═══ CELL 9: Download Weights ═══
import shutil
for model_name in MODELS:
    run_name = model_name.replace('.pt', '') + '_ood_17cls'
    best = Path(f'/kaggle/working/runs/detect/{run_name}/weights/best.pt')
    if best.exists():
        dst = Path(f'/kaggle/working/{run_name}_best.pt')
        shutil.copy2(best, dst)
        print(f'✅ {dst.name} ({dst.stat().st_size / 1e6:.1f} MB)')

for f in [RESULTS_CSV, PERCLASS_CSV]:
    if Path(f).exists(): print(f'✅ {f}')